Los repositorios y datasets oficiales para empezar a experimentar son:Para Grafos y Comunidades:TwiBot-22 / TwiBot-20: Archivo del repositorio en GitHub (BunsenFeng/TwiBot-20) y ficha técnica en Botometer Repository.Cresci-2017: Disponible en Kaggle (hashemalsalmi/cresci-2017) y el subset de pruebas en GitHub (ashu273k/TrustNet).  Para Texto y Similitud Semántica:MGTBench: Disponible en GitHub (MBZUAI-NLP/MGTBench).

In [1]:
!pip install pyvis pandas sentence-transformers scikit-learn seaborn matplotlib fastapi uvicorn community python-louvain kagglehub

  Using cached community-1.0.0b1-py3-none-any.whl


In [3]:
!pip uninstall -y community python-louvain
!pip install python-louvain

Found existing installation: python-louvain 0.16
Uninstalling python-louvain-0.16:
  Successfully uninstalled python-louvain-0.16
  Using cached python_louvain-0.16-py3-none-any.whl


In [5]:
import sys, community
print(sys.executable)
print(community.__file__)
print(hasattr(community, "best_partition"))

/home/adalisa/Documentos/proyectos_git/proyecto_botnet/venv_bot/bin/python
/home/adalisa/Documentos/proyectos_git/proyecto_botnet/venv_bot/lib/python3.12/site-packages/community/__init__.py
True


In [4]:
import re
from pathlib import Path
from typing import Dict, List, Any, Optional

import numpy as np
import pandas as pd
import networkx as nx

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix

# ------------------------------------------------------------------------------
# 1. RESOLUCIÓN DINÁMICA DE RUTAS
# ------------------------------------------------------------------------------
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
DATA_DIR = PROJECT_ROOT / "data"

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Directorio de datos: {DATA_DIR}")

# ------------------------------------------------------------------------------
# 2. FUNCIONES AUXILIARES (TTR, MENCIONES Y POISSON)
# ------------------------------------------------------------------------------
def extract_mention(text: Any) -> Optional[str]:
    if pd.isna(text):
        return None
    mentions = re.findall(r'@(\w+)', str(text))
    return mentions[0] if mentions else None


def calculate_ttr(text_series: pd.Series) -> float:
    tokens = " ".join(text_series.dropna().astype(str)).lower().split()
    if not tokens:
        return 0.0
    return len(set(tokens)) / len(tokens)


def calculate_poisson_anomaly_ratio(timestamps: List[float]) -> float:
    valid_ts = [t for t in timestamps if pd.notna(t)]
    if len(valid_ts) < 3:
        return 0.0

    iats = np.diff(np.sort(valid_ts))
    mean_iat = np.mean(iats)
    std_iat = np.std(iats)

    if mean_iat == 0 or std_iat == 0:
        return 1.0

    z_scores = np.abs((iats - mean_iat) / std_iat)
    anomalous_iats = np.sum((z_scores > 3.0) | (iats < 1.0))
    return float(anomalous_iats / len(iats))


def to_epoch_seconds(series: pd.Series) -> pd.Series:
    """Convierte datetimes (tz-aware UTC) a segundos epoch, independiente de la resolución."""
    return (series - pd.Timestamp("1970-01-01", tz="UTC")) // pd.Timedelta(seconds=1)

# ------------------------------------------------------------------------------
# 3. CARGA DE DATASETS LOCALES Y GROUND TRUTH
# ------------------------------------------------------------------------------
dataset_sources = [
    {"folder": "genuine_accounts.csv", "is_bot": 0},
    {"folder": "social_spambots_1.csv", "is_bot": 1},
]

all_tweets_dfs = []
user_metadata_map: Dict[str, Dict[str, Any]] = {}

for source in dataset_sources:
    folder_path = DATA_DIR / source["folder"]
    is_bot_label = source["is_bot"]

    if not folder_path.exists():
        print(f"Advertencia: No se encontró la carpeta {folder_path}")
        continue

    # A) Leer users.csv
    users_csv = folder_path / "users.csv"
    if users_csv.exists():
        try:
            u_df = pd.read_csv(users_csv, low_memory=False)
            id_col = next((c for c in ['id', 'user_id'] if c in u_df.columns), None)
            created_col = next((c for c in ['created_at', 'user_created_at'] if c in u_df.columns), None)

            if id_col:
                if created_col:
                    created_dt = pd.to_datetime(u_df[created_col], errors='coerce', utc=True, format='mixed')
                    created_ts = to_epoch_seconds(created_dt).astype('float64')
                else:
                    created_ts = pd.Series(np.nan, index=u_df.index)

                for u_id, c_ts in zip(u_df[id_col].astype(str), created_ts):
                    user_metadata_map[u_id] = {
                        'created_ts': float(c_ts) if pd.notna(c_ts) else np.nan,
                        'is_bot_truth': is_bot_label,
                    }
        except Exception as e:
            print(f"Error procesando {users_csv}: {e}")

    # B) Leer tweets.csv
    tweets_csv = folder_path / "tweets.csv"
    if tweets_csv.exists():
        try:
            t_df = pd.read_csv(tweets_csv, low_memory=False, encoding='utf-8', on_bad_lines='skip')
        except Exception:
            t_df = pd.read_csv(tweets_csv, low_memory=False, encoding='latin1', on_bad_lines='skip')

        user_col = next((c for c in ['user_id', 'user_id_str', 'user', 'author', 'id'] if c in t_df.columns), None)
        text_col = next((c for c in ['text', 'full_text', 'tweet', 'content'] if c in t_df.columns), None)
        time_col = next((c for c in ['created_at', 'timestamp', 'date_time'] if c in t_df.columns), None)

        if user_col and text_col and time_col:
            sub_df = t_df[[user_col, text_col, time_col]].dropna()
            sub_df = sub_df.rename(columns={user_col: 'user_id', text_col: 'text', time_col: 'created_at'})
            sub_df['is_bot_truth'] = is_bot_label
            all_tweets_dfs.append(sub_df)

if not all_tweets_dfs:
    raise FileNotFoundError("No se pudieron cargar archivos de tweets desde data/")

df = pd.concat(all_tweets_dfs, ignore_index=True)
df['user_id_str'] = df['user_id'].astype(str)
df['mentioned_user'] = df['text'].apply(extract_mention)
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce', utc=True, format='mixed')
df = df.dropna(subset=['created_at', 'text', 'user_id'])
df['timestamp'] = to_epoch_seconds(df['created_at'])

# Ordenamiento cronológico Point-in-Time (PIT)
df = df.sort_values('created_at').reset_index(drop=True)

# Ventana de corte PIT
cutoff_time = df['created_at'].quantile(0.80)
df_pit = df[df['created_at'] <= cutoff_time].copy()
if len(df_pit) > 10000:
    df_pit = df_pit.tail(10000).copy()

print(f"Total de registros cargados en ventana PIT: {len(df_pit)}")

# ------------------------------------------------------------------------------
# 4. CAPA 1: GRAFO Y COMUNIDADES (LOUVAIN con NetworkX, sin paquete externo)
# ------------------------------------------------------------------------------
G = nx.Graph()
for u_src, u_dst in zip(df_pit['user_id_str'], df_pit['mentioned_user']):
    u_src = str(u_src)
    G.add_node(u_src)
    if pd.notna(u_dst):
        u_dst = str(u_dst)
        if u_dst != 'None' and u_dst != u_src:
            G.add_edge(u_src, u_dst)

communities = nx.community.louvain_communities(G, seed=42)
partition: Dict[str, int] = {node: cid for cid, comm in enumerate(communities) for node in comm}

print(f"Nodos: {G.number_of_nodes()} | Aristas: {G.number_of_edges()} | Comunidades: {len(communities)}")

# ------------------------------------------------------------------------------
# 5. CAPA 2: EMBEDDINGS SEMÁNTICOS Y VECTORES DE USUARIO
# ------------------------------------------------------------------------------
print("Calculando embeddings de texto con MiniLM...")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
text_embeddings = model.encode(df_pit['text'].astype(str).tolist(), show_progress_bar=False, batch_size=256)
df_pit['embedding'] = list(text_embeddings)

user_embeddings: Dict[str, np.ndarray] = {}
user_timestamps: Dict[str, List[float]] = {}

for user_id, group in df_pit.groupby('user_id_str'):
    user_embeddings[user_id] = np.mean(np.array(group['embedding'].tolist()), axis=0)
    user_timestamps[user_id] = group['timestamp'].sort_values().tolist()

embedding_dim = text_embeddings.shape[1]
all_users = list(partition.keys())

for u in all_users:
    if u not in user_embeddings:
        user_embeddings[u] = np.zeros(embedding_dim)
        user_timestamps[u] = []

user_matrix = np.array([user_embeddings[u] for u in all_users])
user_to_idx = {u: idx for idx, u in enumerate(all_users)}

norms = np.linalg.norm(user_matrix, axis=1, keepdims=True)
norms[norms == 0] = 1e-10
sim_matrix = cosine_similarity(user_matrix / norms)

# ------------------------------------------------------------------------------
# 6. EXTRACCIÓN DE MÉTRICAS MULTI-CAPA POR COMUNIDAD
# ------------------------------------------------------------------------------
features_list = []
community_metadata = []

for comm_id in set(partition.values()):
    users_in_comm = [u for u, c in partition.items() if c == comm_id]
    if len(users_in_comm) > 1:
        # Ground Truth de la comunidad (mayoría)
        comm_truth_labels = [
            user_metadata_map[u]['is_bot_truth']
            for u in users_in_comm
            if u in user_metadata_map and 'is_bot_truth' in user_metadata_map[u]
        ]
        ground_truth_comm = int(np.mean(comm_truth_labels) >= 0.5) if comm_truth_labels else 0

        # A) Capa Semántica
        indices = [user_to_idx[u] for u in users_in_comm]
        sub_matrix = sim_matrix[np.ix_(indices, indices)]
        mask = ~np.eye(sub_matrix.shape[0], dtype=bool)
        avg_sim = float(sub_matrix[mask].mean()) if mask.any() else 0.0

        # B) Capa Grafo
        sub_graph = G.subgraph(users_in_comm)
        density = float(nx.density(sub_graph))

        # C) Capa Telemetría Temporal & Poisson
        comm_iat_std = []
        comm_poisson_ratios = []
        for u in users_in_comm:
            ts = user_timestamps[u]
            if len(ts) > 1:
                comm_iat_std.append(float(np.std(np.diff(ts))))
            else:
                comm_iat_std.append(300.0)
            comm_poisson_ratios.append(calculate_poisson_anomaly_ratio(ts))

        avg_iat_std = float(np.mean(comm_iat_std))
        poisson_ratio = float(np.mean(comm_poisson_ratios))

        # D) Sincronía en Creación de Cuentas (Horas)
        creation_ts = [
            user_metadata_map[u]['created_ts']
            for u in users_in_comm
            if u in user_metadata_map and pd.notna(user_metadata_map[u]['created_ts'])
        ]
        creation_std_hours = (np.std(creation_ts) / 3600.0) if len(creation_ts) > 1 else 8760.0

        # E) Rigidez Léxica (TTR)
        comm_texts = df_pit[df_pit['user_id_str'].isin(users_in_comm)]['text']
        ttr_score = calculate_ttr(comm_texts)

        features_list.append([
            density,
            avg_sim,
            avg_iat_std,
            poisson_ratio,
            creation_std_hours,
            ttr_score,
        ])

        community_metadata.append({
            'comm_id': comm_id,
            'users_count': len(users_in_comm),
            'ground_truth': ground_truth_comm,
            'density': density,
            'avg_sim': avg_sim,
            'avg_iat_std': avg_iat_std,
            'poisson_ratio': poisson_ratio,
            'creation_std_hours': creation_std_hours,
            'ttr_score': ttr_score,
        })

if not features_list:
    raise ValueError("No se formaron comunidades con más de 1 usuario. Revisa las menciones/datos.")

X_real = np.array(features_list)

# ------------------------------------------------------------------------------
# 7. MOTOR DE CUADRANTES ESPACIALES 2D (IF + LOF ESTANDARIZADO)
# ------------------------------------------------------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_real)

n_samples = len(X_scaled)
n_neighbors_lof = max(1, min(15, n_samples - 1))

# Isolation Forest (aislamiento global)
if_model = IsolationForest(contamination=0.15, random_state=42)
if_model.fit(X_scaled)
raw_if = -if_model.decision_function(X_scaled)

# Local Outlier Factor (densidad local) - modo outlier detection sobre el mismo set
lof_model = LocalOutlierFactor(n_neighbors=n_neighbors_lof, novelty=False)
lof_model.fit_predict(X_scaled)
raw_lof = -lof_model.negative_outlier_factor_


def minmax(v: np.ndarray) -> np.ndarray:
    if len(v) < 2 or np.ptp(v) == 0:
        return np.zeros_like(v, dtype=float)
    return MinMaxScaler().fit_transform(v.reshape(-1, 1)).flatten()


scores_if = minmax(raw_if)
scores_lof = minmax(raw_lof)

pred_labels = []
quadrants = []

for s_if, s_lof in zip(scores_if, scores_lof):
    is_if_anom = s_if >= 0.50
    is_lof_anom = s_lof >= 0.50

    if is_if_anom and is_lof_anom:
        quadrant, pred = "I: BOTNET_MASIVA_CRITICA", 1
    elif not is_if_anom and is_lof_anom:
        quadrant, pred = "II: ANOMALIA_LOCAL_INFILTRADA", 1
    elif is_if_anom and not is_lof_anom:
        quadrant, pred = "III: ANOMALIA_GLOBAL_DISPERSA", 1
    else:
        quadrant, pred = "IV: HUMANO_ORGANICO", 0

    quadrants.append(quadrant)
    pred_labels.append(pred)

# ------------------------------------------------------------------------------
# 8. CONSOLIDACIÓN Y EVALUACIÓN DE MÉTRICAS
# ------------------------------------------------------------------------------
comm_df = pd.DataFrame(community_metadata)
comm_df['Pred_Bot'] = pred_labels
comm_df['Cuadrante'] = quadrants

y_true = comm_df['ground_truth'].values
y_pred = comm_df['Pred_Bot'].values

print("\n--- INFORME DE EVALUACIÓN SOBRE DATOS REALES (LOCAL) ---")
print(classification_report(
    y_true, y_pred,
    labels=[0, 1],
    target_names=['HUMANO (0)', 'BOT (1)'],
    zero_division=0,
))

print("\nMatriz de Confusión:")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

/home/adalisa/Documentos/proyectos_git/proyecto_botnet/venv_bot/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Raíz del proyecto: /home/adalisa/Documentos/proyectos_git/proyecto_botnet
Directorio de datos: /home/adalisa/Documentos/proyectos_git/proyecto_botnet/data
Total de registros cargados en ventana PIT: 10000
Nodos: 4199 | Aristas: 3749 | Comunidades: 542
Calculando embeddings de texto con MiniLM...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1651.58it/s]



--- INFORME DE EVALUACIÓN SOBRE DATOS REALES (LOCAL) ---
              precision    recall  f1-score   support

  HUMANO (0)       1.00      0.93      0.96       433
     BOT (1)       0.00      0.00      0.00         0

    accuracy                           0.93       433
   macro avg       0.50      0.46      0.48       433
weighted avg       1.00      0.93      0.96       433


Matriz de Confusión:
[[401  32]
 [  0   0]]


/home/adalisa/Documentos/proyectos_git/proyecto_botnet/venv_bot/lib/python3.12/site-packages/sklearn/neighbors/_lof.py:327: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(
